In [ ]:
import os
import polars as pl

experiment = "new"
model_types = ["lr","dt","rf","lgb","mlp"]

metrics = ["AUC", "F1", "precision", "recall", "accuracy", "accuracy_train"]

hyperparams = {
    "rf":  ["rf_n_estimators", "rf_max_depth", "rf_min_samples_leaf"],
    "lgb": ["lgb_n_estimators", "lgb_max_depth", "learning_rate", "min_child_weight",
             "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"],
    "mlp": ["hidden_sizes", "mlp_learning_rate", "dropout_rate",
             "weight_decay", "batch_size"],
    "dt":  ["dt_max_depth", "dt_criterion", "dt_min_samples_split", "dt_min_samples_leaf"],
    "lr":  ["lr_C", "lr_penalty"],
}

# Metrica primaria + due di tie-breaking (None per disabilitare)
sort_metric   = "F1"
sort_metric_2 = "AUC"
sort_metric_3 = "accuracy"

results_dir = f"data/results/{experiment}"
output_path = f"data/results/{experiment}/summary.txt"
results = {}
lines = []

for model in model_types:
    path = os.path.join(results_dir, f"{model}.csv")
    df = pl.read_csv(path)

    df = df.filter(
        (pl.col("State") == "finished") & pl.col(sort_metric).is_not_null()
    )

    cols = hyperparams[model] + metrics
    df_selected = df.select(["Name"] + cols)

    # Arrotondamento PRIMA dell'ordinamento
    df_selected = df_selected.with_columns(
        [pl.col(m).cast(pl.Float64, strict=False).round(2) for m in metrics]
    )

    # Ordinamento con tie-breaking opzionale
    sort_cols = [m for m in [sort_metric, sort_metric_2, sort_metric_3] if m is not None]
    df_selected = df_selected.sort(sort_cols, descending=True)

    results[model] = df_selected
    top = df_selected.row(0, named=True)

    sort_label = " > ".join(
        m for m in [sort_metric, sort_metric_2, sort_metric_3] if m is not None
    )
    header = f"\n{'='*40}\n {model.upper()} — best by {sort_label} ({top['Name']})\n{'='*40}"
    print(header)
    lines.append(header)

    print("\n  Hyperparameters:")
    lines.append("\n  Hyperparameters:")
    for h in hyperparams[model]:
        line = f"    {h:25s} {top[h]}"
        print(line)
        lines.append(line)

    print("\n  Metrics:")
    lines.append("\n  Metrics:")
    for m in metrics:
        line = f"    {m:25s} {top[m]}"
        print(line)
        lines.append(line)

with open(output_path, "w") as f:
    f.write("\n".join(lines))

print(f"\n\nSaved to {output_path}")

In [ ]:
model = "lgb"
run_name = "lilac-sweep-23"

path = os.path.join(results_dir, f"{model}.csv")
df = pl.read_csv(path)

df_run = df.filter(pl.col("Name") == run_name)

if df_run.is_empty():
    print(f"Run '{run_name}' not found in {model}")
else:
    row = df_run.row(0, named=True)
    print(f"\n{'='*40}\n {model.upper()} — run: {run_name}\n{'='*40}")

    print("\n  Hyperparameters:")
    for h in hyperparams[model]:
        print(f"    {h:25s} {row.get(h, 'N/A')}")

    print("\n  Metrics:")
    for m in metrics:
        val = row.get(m)
        if val is not None:
            try:
                val = round(float(val), 2)
            except (ValueError, TypeError):
                pass
        print(f"    {m:25s} {val}")

In [5]:
# --- Compare with another experiment's all.csv ---
other_label = "No-competitors"  
other_path = f"data/results/{other_label}/all.csv"  # <-- change this path as needed
current_label = experiment

# Read the other experiment's results and pick best F1 per model
df_other = pl.read_csv(other_path)
df_other = df_other.filter(
    (pl.col("State") == "finished") & pl.col(sort_metric).is_not_null()
)
best_other = (
    df_other
    .group_by("model_type")
    .agg([pl.col(m).cast(pl.Float64, strict=False).max().round(2).alias(m) for m in metrics])
    .sort("model_type")
)

# Build best-per-model from current experiment
rows = []
for model, df_selected in results.items():
    top = df_selected.row(0, named=True)
    row = {"model_type": model}
    for m in metrics:
        row[m] = top[m]
    rows.append(row)
best_current = pl.DataFrame(rows).sort("model_type")

# Compare side by side
print(f"{'Model':<6} {'Metric':<20} {current_label:>15} {other_label:>15} {'Diff':>10} {'Diff%':>10}")
print("-" * 80)

for model in sorted(set(best_current["model_type"]) & set(best_other["model_type"])):
    cur_row = best_current.filter(pl.col("model_type") == model).row(0, named=True)
    oth_row = best_other.filter(pl.col("model_type") == model).row(0, named=True)
    for m in metrics:
        cur_val = cur_row[m] if cur_row[m] is not None else float("nan")
        oth_val = oth_row[m] if oth_row[m] is not None else float("nan")
        valid = cur_val == cur_val and oth_val == oth_val
        diff = round(oth_val -cur_val, 2) if valid else float("nan")
        pct = round((oth_val - cur_val ) / cur_val * 100, 1) if valid and cur_val != 0 else float("nan")
        sign_d = "+" if diff > 0 else ""
        sign_p = "+" if pct > 0 else ""
        pct_str = f"{sign_p}{pct}%" if pct == pct else "nan"
        print(f"{model:<6} {m:<20} {cur_val:>15} {oth_val:>15} {sign_d + str(diff):>10} {pct_str:>10}")
    print()


Model  Metric                           new  No-competitors       Diff      Diff%
--------------------------------------------------------------------------------
dt     AUC                             0.79            0.77      -0.02      -2.5%
dt     F1                              0.62            0.61      -0.01      -1.6%
dt     precision                       0.52            0.56      +0.04      +7.7%
dt     recall                          0.77            0.68      -0.09     -11.7%
dt     accuracy                         0.7            0.72      +0.02      +2.9%
dt     accuracy_train                   0.7            0.72      +0.02      +2.9%

lgb    AUC                             0.81             0.8      -0.01      -1.2%
lgb    F1                              0.64            0.63      -0.01      -1.6%
lgb    precision                       0.57            0.56      -0.01      -1.8%
lgb    recall                          0.71            0.71        0.0       0.0%
lgb    accuracy 